# Download and clean ERA5

Downloads daily ERA5 pressure-level data for every month with a GridMET storm, then merges it and computes integrated vapor transport (IVT).

**Part 1: Download** *(slow; limited by the CDS queue)*
1. Find every month with at least one storm day (from `storm_days.nc4`).
2. Delete any earlier downloads whose grid doesn't match the current study-area box.
3. Download daily means of geopotential, temperature, specific humidity and u/v wind at 600, 700, 800 and 900 hPa over the study area, with days in UTC−7 to match GridMET. Files that are already downloaded are skipped, so this step can be restarted if it stops partway.

**Part 2: Merge and IVT** *(lots of time and memory; can run separately from Part 1)*
1. Merge all downloads into one dataset and convert geopotential to geopotential height [m].
2. Compute IVT from 600–900 hPa using daily-mean q, u and v:
   IVT = |(1/g) ∫ q **V** dp|  [kg m⁻¹ s⁻¹]

**Inputs**
- `Data/Cleaned/GridMET/storm_days.nc4`: from the GridMET storm notebook
- `config.py`: `data_folder`, `shapes`, `basin_names_list`
- A CDS API key (`~/.cdsapirc`)

**Outputs**
| File | Contents |
|------|----------|
| `Data/ERA5/era5_download_<var>_<year>_<month>.nc4` | Raw monthly downloads |
| `Data/Cleaned/ERA5/era5_complete_ivt.nc4` | All variables (pressure in hPa) plus `ivt` |

Note: this IVT covers 600–900 hPa only and is computed from daily means, so it is lower than standard 1000–300 hPa IVT from sub-daily data.

In [ ]:
# %% Imports and study area
import glob
import os

import cdsapi
import pandas as pd
import xarray as xr

from config import data_folder, basin_names_list, shapes

minx, miny, maxx, maxy = shapes[shapes['name'].isin(basin_names_list)].total_bounds
AREA = [maxy, minx, miny, maxx]  # CDS order: North, West, South, East

ERA5_VARS = ['geopotential', 'temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind']
era5_raw = os.path.join(data_folder, 'Data/ERA5')


def matches_area(path, tol=0.25):
    """True if a downloaded file's grid edges are within one ERA5 cell (0.25°) of AREA."""
    with xr.open_dataset(path) as d:
        edges = [d.latitude.max(), d.longitude.min(), d.latitude.min(), d.longitude.max()]
        return all(abs(float(e) - a) <= tol for e, a in zip(edges, AREA))

In [ ]:
# %% Months with at least one GridMET storm day
days = xr.open_dataset(os.path.join(data_folder, 'Data/Cleaned/GridMET/storm_days.nc4'))['Day']
storm_months = sorted({(d.year, d.month) for d in pd.to_datetime(days.values)})
len(storm_months)

In [ ]:
# %% Delete downloads whose grid doesn't match AREA (e.g. from an older, wider box).
#    The download cell below re-fetches them.
bad = [f for f in glob.glob(os.path.join(era5_raw, 'era5_download_*.nc4')) if not matches_area(f)]
for f in bad:
    os.remove(f)
print(f'removed {len(bad)} mismatched files')

In [ ]:
# %% Download ERA5 daily means for every storm month (skips files already downloaded)
REQUEST = {
    'product_type': 'reanalysis',
    'day': [f'{d:02d}' for d in range(1, 32)],
    'pressure_level': ['600', '700', '800', '900'],
    'daily_statistic': 'daily_mean',
    'time_zone': 'utc-07:00',  # to match GridMET days
    'frequency': '6_hourly',
    'area': AREA,
    'format': 'netcdf',
}

client = cdsapi.Client()
for year, month in storm_months:
    for var in ERA5_VARS:
        target = os.path.join(era5_raw, f'era5_download_{var}_{year}_{month:02d}.nc4')
        if not os.path.isfile(target):
            client.retrieve('derived-era5-pressure-levels-daily-statistics',
                            {**REQUEST, 'variable': var, 'year': str(year), 'month': f'{month:02d}'},
                            target)

In [ ]:
## ===== Part 2: merge downloads and compute IVT (can run separately from Part 1) =====
import gc
import glob
import os

import numpy as np
import xarray as xr

from config import data_folder

ERA5_VARS = ['geopotential', 'temperature', 'specific_humidity', 'u_component_of_wind', 'v_component_of_wind']
era5_raw = os.path.join(data_folder, 'Data/ERA5')
era5_out = os.path.join(data_folder, 'Data/Cleaned/ERA5')
G = 9.80665  # m s^-2

In [ ]:
# %% Merge all downloads into one temporary file (much faster to work from than hundreds of files)
files = [f for var in ERA5_VARS for f in glob.glob(os.path.join(era5_raw, f'era5_download_{var}_*.nc4'))]
ds = xr.open_mfdataset(files, engine='h5netcdf', parallel=True).drop_vars('number')
ds['z'] = (ds['z'] / G).assign_attrs(units='m')  # geopotential -> geopotential height

for v in ds.variables:
    ds[v].encoding = {}  # per-file encodings conflict when writing the merged file

merged_path = os.path.join(era5_out, 'era5_complete.nc4')
ds.to_netcdf(merged_path)
del ds
gc.collect()

In [ ]:
# %% IVT = |(1/g) ∫ q·V dp| over the downloaded pressure levels (needs lots of time + memory)
ds = xr.open_dataset(merged_path).sortby('pressure_level')

# Integrate in Pa, but keep pressure_level in hPa in the saved file
to_pa = 100 if ds['pressure_level'].max() < 2000 else 1
qv = ds[['q', 'u', 'v']].assign_coords(pressure_level=ds['pressure_level'] * to_pa)
ivt_u = (qv['q'] * qv['u']).integrate('pressure_level') / G
ivt_v = (qv['q'] * qv['v']).integrate('pressure_level') / G
ds['ivt'] = np.hypot(ivt_u, ivt_v).assign_attrs(units='kg m^-1 s^-1', long_name='Integrated Vapor Transport')

ds.to_netcdf(os.path.join(era5_out, 'era5_complete_ivt.nc4'))
ds.close()
os.remove(merged_path)  # era5_complete_ivt.nc4 holds everything in it, plus IVT